In [5]:
import pandas as pd

trad_spam1_tweets = pd.read_csv('traditional_spambots_1.csv/traditional_spambots_1.csv/tweets.csv', nrows=50000)
print(trad_spam1_tweets.shape)
print(trad_spam1_tweets.head())

(50000, 25)
            id                                               text  \
0  22642586115     CPPRI Recruitment 2010 at http://ping.fm/yp8zH   
1  22642583483  National Games Secretariat Recruitment 2010  :...   
2  22642524678     CIPET Recruitment Jobs at http://ping.fm/KnFCa   
3  22642504361      DIAT Recruitment 2010 at http://ping.fm/huS9m   
4  22642475789       BHEL Recruitment 2010 : http://ping.fm/PLWWA   

                                              source  user_id  truncated  \
0  <a href="http://www.ping.fm/" rel="nofollow">P...  7248952        NaN   
1  <a href="http://www.ping.fm/" rel="nofollow">P...  7248952        NaN   
2  <a href="http://www.ping.fm/" rel="nofollow">P...  7248952        NaN   
3  <a href="http://www.ping.fm/" rel="nofollow">P...  7248952        NaN   
4  <a href="http://www.ping.fm/" rel="nofollow">P...  7248952        NaN   

   in_reply_to_status_id  in_reply_to_user_id in_reply_to_screen_name  \
0                      0                   

In [14]:
import pandas as pd

# Only folders WITH tweets.csv
folders = {
    'genuine_accounts.csv': 0,
    'traditional_spambots_1.csv': 1,
    'social_spambots_1.csv': 1,
    'social_spambots_2.csv': 1,
    'social_spambots_3.csv': 1,
    'fake_followers.csv': 1,
}

all_tweets = []

for folder, label in folders.items():
    path = f'{folder}/{folder}/tweets.csv'
    df = pd.read_csv(path, nrows=50000, encoding='latin1')
    df['label'] = label
    df['account_type'] = folder
    all_tweets.append(df)
    print(f'{folder}: loaded {df.shape[0]} rows')

combined_tweets = pd.concat(all_tweets, ignore_index=True)
print('Combined shape:', combined_tweets.shape)
# Fill missing tweets with empty string first
combined_tweets['text'] = combined_tweets['text'].fillna('')

# Now calculate tweet length safely
combined_tweets['tweet_length'] = combined_tweets['text'].astype(str).apply(len)

print(combined_tweets[['user_id', 'text', 'tweet_length']].head())

genuine_accounts.csv: loaded 50000 rows
traditional_spambots_1.csv: loaded 50000 rows


C:\Users\almas\AppData\Local\Temp\ipykernel_26376\3717429876.py:17: DtypeWarning: Columns (0: place) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, nrows=50000, encoding='latin1')


social_spambots_1.csv: loaded 50000 rows
social_spambots_2.csv: loaded 50000 rows
social_spambots_3.csv: loaded 50000 rows
fake_followers.csv: loaded 50000 rows
Combined shape: (300000, 27)
   user_id                                               text  tweet_length
0   678033  RT @morningJewshow: Speaking about Jews and co...           129
1   678033  This age/face recognition thing..no reason pla...           140
2   678033  Only upside of the moment I can think of is th...           119
3   678033  If you're going to think about+create experien...           140
4   678033  Watching a thread on FB about possible future ...           133


In [16]:
# First, add helper columns per tweet
combined_tweets['has_url'] = combined_tweets['text'].str.contains('http', case=False, na=False)
combined_tweets['has_hashtag'] = combined_tweets['text'].str.contains('#', na=False)

# Now group by user_id to get one row per account
text_features = combined_tweets.groupby('user_id').agg(
    avg_tweet_length=('tweet_length', 'mean'),
    tweet_count=('text', 'count'),
    pct_with_url=('has_url', 'mean'),
    pct_with_hashtag=('has_hashtag', 'mean'),
    unique_tweet_ratio=('text', lambda x: x.nunique() / len(x))
).reset_index()

# Add back the label (bot/human) - taking the first label seen per user
labels = combined_tweets.groupby('user_id')['label'].first().reset_index()
text_features = text_features.merge(labels, on='user_id')

print(text_features.shape)
print(text_features.head())
text_features.to_csv('text_features.csv', index=False)
print("Saved!")

(508, 7)
   user_id  avg_tweet_length  tweet_count  pct_with_url  pct_with_hashtag  \
0   678033        110.966197         3195      0.316745          0.267293   
1   722623         85.065917         3201      0.254608          0.268666   
2   755116         82.521969         3209      0.256155          0.028358   
3   755746         85.584416         3234      0.493816          0.239951   
4   785080         96.123957         3235      0.099845          0.040185   

   unique_tweet_ratio  label  
0            0.992488      0  
1            0.998438      0  
2            1.000000      0  
3            0.998454      0  
4            1.000000      0  
Saved!


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

feature_columns = ['avg_tweet_length', 'tweet_count', 'pct_with_url', 'pct_with_hashtag', 'unique_tweet_ratio']

X = text_features[feature_columns]
y = text_features['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.9607843137254902
              precision    recall  f1-score   support

           0       0.33      0.50      0.40         4
           1       0.99      0.97      0.98       149

    accuracy                           0.96       153
   macro avg       0.66      0.74      0.69       153
weighted avg       0.97      0.96      0.96       153



In [18]:
print(text_features['label'].value_counts())

label
1    490
0     18
Name: count, dtype: int64


In [19]:
# Load a much bigger sample from genuine_accounts specifically
genuine_tweets = pd.read_csv('genuine_accounts.csv/genuine_accounts.csv/tweets.csv', 
                               nrows=300000, encoding='latin1')
genuine_tweets['label'] = 0
genuine_tweets['account_type'] = 'genuine_accounts.csv'

print("Unique genuine accounts:", genuine_tweets['user_id'].nunique())

Unique genuine accounts: 106


In [20]:
# Load an even bigger sample from genuine_accounts specifically
genuine_tweets = pd.read_csv('genuine_accounts.csv/genuine_accounts.csv/tweets.csv', 
                               nrows=600000, encoding='latin1')
genuine_tweets['label'] = 0
genuine_tweets['account_type'] = 'genuine_accounts.csv'

print("Unique genuine accounts:", genuine_tweets['user_id'].nunique())

Unique genuine accounts: 213


In [21]:
import pandas as pd

# Folders with tweets.csv, using bigger sample for genuine_accounts
folders_nrows = {
    'genuine_accounts.csv': (0, 600000),
    'traditional_spambots_1.csv': (1, 50000),
    'social_spambots_1.csv': (1, 50000),
    'social_spambots_2.csv': (1, 50000),
    'social_spambots_3.csv': (1, 50000),
    'fake_followers.csv': (1, 50000),
}

all_tweets = []

for folder, (label, n) in folders_nrows.items():
    path = f'{folder}/{folder}/tweets.csv'
    df = pd.read_csv(path, nrows=n, encoding='latin1')
    df['label'] = label
    df['account_type'] = folder
    all_tweets.append(df)
    print(f'{folder}: loaded {df.shape[0]} rows, {df["user_id"].nunique()} unique accounts')

combined_tweets = pd.concat(all_tweets, ignore_index=True)
print('Combined shape:', combined_tweets.shape)

genuine_accounts.csv: loaded 600000 rows, 213 unique accounts
traditional_spambots_1.csv: loaded 50000 rows, 123 unique accounts


C:\Users\almas\AppData\Local\Temp\ipykernel_26376\1030187910.py:17: DtypeWarning: Columns (0: place) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, nrows=n, encoding='latin1')


social_spambots_1.csv: loaded 50000 rows, 24 unique accounts
social_spambots_2.csv: loaded 50000 rows, 51 unique accounts
social_spambots_3.csv: loaded 50000 rows, 18 unique accounts
fake_followers.csv: loaded 50000 rows, 274 unique accounts
Combined shape: (850000, 27)


In [23]:
# Fill missing text
combined_tweets['text'] = combined_tweets['text'].fillna('')

# Tweet length
combined_tweets['tweet_length'] = combined_tweets['text'].astype(str).apply(len)

# URL and hashtag presence
combined_tweets['has_url'] = combined_tweets['text'].str.contains('http', case=False, na=False)
combined_tweets['has_hashtag'] = combined_tweets['text'].str.contains('#', na=False)

# Group by account to build one row per user
text_features = combined_tweets.groupby('user_id').agg(
    avg_tweet_length=('tweet_length', 'mean'),
    tweet_count=('text', 'count'),
    pct_with_url=('has_url', 'mean'),
    pct_with_hashtag=('has_hashtag', 'mean'),
    unique_tweet_ratio=('text', lambda x: x.nunique() / len(x))
).reset_index()

# Add back label
labels = combined_tweets.groupby('user_id')['label'].first().reset_index()
text_features = text_features.merge(labels, on='user_id')

print(text_features.shape)
print(text_features['label'].value_counts())
# Save the updated text features
text_features.to_csv('text_features.csv', index=False)
print("Saved!")

(703, 7)
label
1    490
0    213
Name: count, dtype: int64
Saved!


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

feature_columns = ['avg_tweet_length', 'tweet_count', 'pct_with_url', 'pct_with_hashtag', 'unique_tweet_ratio']

X = text_features[feature_columns]
y = text_features['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.933649289099526
              precision    recall  f1-score   support

           0       0.88      0.91      0.89        64
           1       0.96      0.95      0.95       147

    accuracy                           0.93       211
   macro avg       0.92      0.93      0.92       211
weighted avg       0.93      0.93      0.93       211



In [25]:
# Combine all tweets per account into one text blob
account_text = combined_tweets.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
account_text.columns = ['user_id', 'all_text']

# Add back the label
account_text = account_text.merge(labels, on='user_id')

print(account_text.shape)
print(account_text.head())

(703, 3)
   user_id                                           all_text  label
0   678033  RT @morningJewshow: Speaking about Jews and co...      0
1   722623  @eegees oh please oh please oh please tell me ...      0
2   755116  è http://t.co/Ap9bdtACCU @lenawash fuck ze G...      0
3   755746  Pinch me I must be dreaming ... Enjoying my lo...      0
4   785080  RT @Popehat: Update: Redditor "Scalia_FuckedYo...      0


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text into TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = vectorizer.fit_transform(account_text['all_text'])

print(X_tfidf.shape)

(703, 5000)


In [ ]:
from sklearn.model_selection import train_test_split

# Split BEFORE vectorizing, so TF-IDF never sees test text
X_train_text, X_test_text, y_train, y_test = train_test_split(
    account_text['all_text'], 
    account_text['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=account_text['label']
)

print(X_train_text.shape, X_test_text.shape)


vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train_text)   # fit + transform on train
X_test_tfidf = vectorizer.transform(X_test_text)          # transform only on test

print(X_train_tfidf.shape, X_test_tfidf.shape)

(562,) (141,)
(562, 5000) (141, 5000)


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.9432624113475178
              precision    recall  f1-score   support

           0       0.91      0.91      0.91        43
           1       0.96      0.96      0.96        98

    accuracy                           0.94       141
   macro avg       0.93      0.93      0.93       141
weighted avg       0.94      0.94      0.94       141



In [30]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.950354609929078
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        43
           1       0.97      0.96      0.96        98

    accuracy                           0.95       141
   macro avg       0.94      0.94      0.94       141
weighted avg       0.95      0.95      0.95       141



In [31]:
vectorizer_20k = TfidfVectorizer(max_features=20000, stop_words='english')

X_train_tfidf_20k = vectorizer_20k.fit_transform(X_train_text)   # same train split as before
X_test_tfidf_20k = vectorizer_20k.transform(X_test_text)

print(X_train_tfidf_20k.shape, X_test_tfidf_20k.shape)

(562, 20000) (141, 20000)


In [32]:
lr_20k = LogisticRegression(max_iter=1000, random_state=42)
lr_20k.fit(X_train_tfidf_20k, y_train)
y_pred_lr_20k = lr_20k.predict(X_test_tfidf_20k)

print("Logistic Regression (20k) Accuracy:", accuracy_score(y_test, y_pred_lr_20k))
print(classification_report(y_test, y_pred_lr_20k))

svm_20k = SVC(kernel='linear', random_state=42)
svm_20k.fit(X_train_tfidf_20k, y_train)
y_pred_svm_20k = svm_20k.predict(X_test_tfidf_20k)

print("SVM (20k) Accuracy:", accuracy_score(y_test, y_pred_svm_20k))
print(classification_report(y_test, y_pred_svm_20k))

Logistic Regression (20k) Accuracy: 0.9432624113475178
              precision    recall  f1-score   support

           0       0.89      0.93      0.91        43
           1       0.97      0.95      0.96        98

    accuracy                           0.94       141
   macro avg       0.93      0.94      0.93       141
weighted avg       0.94      0.94      0.94       141

SVM (20k) Accuracy: 0.9574468085106383
              precision    recall  f1-score   support

           0       0.91      0.95      0.93        43
           1       0.98      0.96      0.97        98

    accuracy                           0.96       141
   macro avg       0.95      0.96      0.95       141
weighted avg       0.96      0.96      0.96       141



In [33]:
vectorizer_bigram = TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1, 2))

X_train_tfidf_bg = vectorizer_bigram.fit_transform(X_train_text)
X_test_tfidf_bg = vectorizer_bigram.transform(X_test_text)

print(X_train_tfidf_bg.shape, X_test_tfidf_bg.shape)

(562, 20000) (141, 20000)


In [34]:
svm_bg = SVC(kernel='linear', random_state=42)
svm_bg.fit(X_train_tfidf_bg, y_train)
y_pred_svm_bg = svm_bg.predict(X_test_tfidf_bg)

print("SVM (bigrams, 20k) Accuracy:", accuracy_score(y_test, y_pred_svm_bg))
print(classification_report(y_test, y_pred_svm_bg))

SVM (bigrams, 20k) Accuracy: 0.9574468085106383
              precision    recall  f1-score   support

           0       0.91      0.95      0.93        43
           1       0.98      0.96      0.97        98

    accuracy                           0.96       141
   macro avg       0.95      0.96      0.95       141
weighted avg       0.96      0.96      0.96       141



In [35]:
import re

def clean_tweet(text):
    text = re.sub(r'^RT\s+@\w+:', '', text)      # remove leading "RT @username:"
    text = re.sub(r'@\w+', '', text)              # remove all @mentions
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()       # collapse extra whitespace
    return text

account_text['all_text_clean'] = account_text['all_text'].apply(clean_tweet)

print(account_text[['all_text', 'all_text_clean']].head())

                                            all_text  \
0  RT @morningJewshow: Speaking about Jews and co...   
1  @eegees oh please oh please oh please tell me ...   
2  è http://t.co/Ap9bdtACCU @lenawash fuck ze G...   
3  Pinch me I must be dreaming ... Enjoying my lo...   
4  RT @Popehat: Update: Redditor "Scalia_FuckedYo...   

                                      all_text_clean  
0  Speaking about Jews and comedy tonight at Temp...  
1  oh please oh please oh please tell me you're c...  
2  è fuck ze Germans. But you can send anything...  
3  Pinch me I must be dreaming ... Enjoying my lo...  
4  Update: Redditor "Scalia_FuckedYourMom" concur...  


In [36]:
X_train_text_c, X_test_text_c, y_train_c, y_test_c = train_test_split(
    account_text['all_text_clean'],
    account_text['label'],
    test_size=0.2,
    random_state=42,
    stratify=account_text['label']
)

vectorizer_clean = TfidfVectorizer(max_features=20000, stop_words='english')
X_train_tfidf_c = vectorizer_clean.fit_transform(X_train_text_c)
X_test_tfidf_c = vectorizer_clean.transform(X_test_text_c)

print(X_train_tfidf_c.shape, X_test_tfidf_c.shape)

(562, 20000) (141, 20000)


In [37]:
svm_clean = SVC(kernel='linear', random_state=42)
svm_clean.fit(X_train_tfidf_c, y_train_c)
y_pred_svm_clean = svm_clean.predict(X_test_tfidf_c)

print("SVM (cleaned, 20k) Accuracy:", accuracy_score(y_test_c, y_pred_svm_clean))
print(classification_report(y_test_c, y_pred_svm_clean))

SVM (cleaned, 20k) Accuracy: 0.9645390070921985
              precision    recall  f1-score   support

           0       0.93      0.95      0.94        43
           1       0.98      0.97      0.97        98

    accuracy                           0.96       141
   macro avg       0.96      0.96      0.96       141
weighted avg       0.96      0.96      0.96       141



In [38]:
import pandas as pd

# Adjust folder paths to match your actual file locations
account_types = {
    'genuine_accounts': 'genuine_accounts/users.csv',
    'traditional_spambots_1': 'traditional_spambots_1/users.csv',
    'social_spambots_1': 'social_spambots_1/users.csv',
    'social_spambots_2': 'social_spambots_2/users.csv',
    'social_spambots_3': 'social_spambots_3/users.csv',
    'fake_followers': 'fake_followers/users.csv',
}

type_frames = []
for type_name, path in account_types.items():
    df = pd.read_csv(path, usecols=['id'])  # just need the user id column
    df['account_type'] = type_name
    df = df.rename(columns={'id': 'user_id'})
    type_frames.append(df)

account_type_map = pd.concat(type_frames, ignore_index=True)
print(account_type_map['account_type'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'genuine_accounts/users.csv'

In [39]:
import os

base_path = r'C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped'
for root, dirs, files in os.walk(base_path):
    for f in files:
        if f == 'users.csv':
            print(os.path.join(root, f))

C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\fake_followers.csv\fake_followers.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\genuine_accounts.csv\genuine_accounts.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\social_spambots_1.csv\social_spambots_1.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\social_spambots_2.csv\social_spambots_2.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\social_spambots_3.csv\social_spambots_3.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\traditional_spambots_1.csv\traditional_spambots_1.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\traditional_spambots_2.csv\traditional_spambots_2.csv\users.csv
C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped\traditional_spambots_3.csv\traditional_spambots_3.csv\users.csv
C:

In [40]:
import pandas as pd

base_path = r'C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped'

account_types = {
    'genuine_accounts': base_path + r'\genuine_accounts.csv\genuine_accounts.csv\users.csv',
    'traditional_spambots_1': base_path + r'\traditional_spambots_1.csv\traditional_spambots_1.csv\users.csv',
    'social_spambots_1': base_path + r'\social_spambots_1.csv\social_spambots_1.csv\users.csv',
    'social_spambots_2': base_path + r'\social_spambots_2.csv\social_spambots_2.csv\users.csv',
    'social_spambots_3': base_path + r'\social_spambots_3.csv\social_spambots_3.csv\users.csv',
    'fake_followers': base_path + r'\fake_followers.csv\fake_followers.csv\users.csv',
}

type_frames = []
for type_name, path in account_types.items():
    df = pd.read_csv(path, usecols=['id'])
    df['account_type'] = type_name
    df = df.rename(columns={'id': 'user_id'})
    type_frames.append(df)

account_type_map = pd.concat(type_frames, ignore_index=True)
print(account_type_map['account_type'].value_counts())

account_type
genuine_accounts          3474
social_spambots_2         3457
fake_followers            3351
traditional_spambots_1    1000
social_spambots_1          991
social_spambots_3          464
Name: count, dtype: int64


In [41]:
account_text_typed = account_text.merge(account_type_map, on='user_id', how='left')

print(account_text_typed['account_type'].value_counts(dropna=False))

account_type
fake_followers            274
genuine_accounts          213
traditional_spambots_1    123
social_spambots_2          51
social_spambots_1          24
social_spambots_3          18
Name: count, dtype: int64


In [42]:
train_mask = account_text_typed['account_type'] != 'fake_followers'
test_mask = account_text_typed['account_type'] == 'fake_followers'

X_train_ct_text = account_text_typed.loc[train_mask, 'all_text_clean']
y_train_ct = account_text_typed.loc[train_mask, 'label']

X_test_ct_text = account_text_typed.loc[test_mask, 'all_text_clean']
y_test_ct = account_text_typed.loc[test_mask, 'label']

print(X_train_ct_text.shape, X_test_ct_text.shape)
print(y_train_ct.value_counts())
print(y_test_ct.value_counts())

(429,) (274,)
label
1    216
0    213
Name: count, dtype: int64
label
1    274
Name: count, dtype: int64


In [43]:
vectorizer_ct = TfidfVectorizer(max_features=20000, stop_words='english')
X_train_ct_tfidf = vectorizer_ct.fit_transform(X_train_ct_text)
X_test_ct_tfidf = vectorizer_ct.transform(X_test_ct_text)

svm_ct = SVC(kernel='linear', random_state=42)
svm_ct.fit(X_train_ct_tfidf, y_train_ct)
y_pred_ct = svm_ct.predict(X_test_ct_tfidf)

print("Cross-type (fake_followers unseen) Accuracy:", accuracy_score(y_test_ct, y_pred_ct))
print(classification_report(y_test_ct, y_pred_ct, zero_division=0))

Cross-type (fake_followers unseen) Accuracy: 0.6934306569343066
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.69      0.82       274

    accuracy                           0.69       274
   macro avg       0.50      0.35      0.41       274
weighted avg       1.00      0.69      0.82       274



In [44]:
import numpy as np

# Get the actual text for the fake_followers test set, aligned with predictions
fake_followers_test = account_text_typed.loc[test_mask].reset_index(drop=True)
fake_followers_test['predicted'] = y_pred_ct

missed = fake_followers_test[fake_followers_test['predicted'] == 0]
caught = fake_followers_test[fake_followers_test['predicted'] == 1]

print(f"Missed (predicted genuine): {len(missed)}")
print(f"Caught (predicted bot): {len(caught)}")
print()
print("Sample of MISSED fake_followers text:")
for text in missed['all_text_clean'].head(5):
    print("-", text[:150])

Missed (predicted genuine): 84
Caught (predicted bot): 190

Sample of MISSED fake_followers text:
- here is the link!!! Well done hubby Two years with my lovely husband - thank you for the cotton flowers Sorry bunny about your ears but I was hungry..
- - Ð¯ ÑÑÑ Ð² ÐÐµÑÐ»Ð¸Ð½ ÐµÐ·Ð´Ð¸Ð», ÐºÑÐ°ÑÐ¾ÑÐ°! ÐÑÐµÐ½Ñ Ð¿Ð¾Ð½ÑÐ°Ð²Ð¸Ð»Ð¾ÑÑ! Ð ÑÑ Ð² ÐÐµÑÐ»Ð¸Ð½Ðµ Ð±ÑÐ»?- ÐÐµ, Ñ Ð½Ðµ Ð±ÑÐ». Ð
- #NoGarpa que se viva enojando por boludeces. RT : #CornudasVirtuales tienen cuernos formados por pixeles RT : #FiestaDeArgentinosChetos Mientras Charl
- I just donated to for his bday and for all the good that NextStep does in this... RT : One of the most compelling talks from #TED. Alan Savory reveals
- Just a little winner !!!! ah sending you a hug xx you piss off !!! Just looked under my bed and realised there is more life under there than inside it


In [45]:
from sklearn.feature_extraction.text import CountVectorizer

# Top words in fake_followers vs top words in other bot types (train set)
other_bots_train = account_text_typed.loc[train_mask & (account_text_typed['label'] == 1), 'all_text_clean']
fake_followers_text = account_text_typed.loc[test_mask, 'all_text_clean']

cv = CountVectorizer(max_features=20, stop_words='english')

top_other = cv.fit_transform(other_bots_train)
print("Top words in TRAINED bot types:", cv.get_feature_names_out())

top_ff = cv.fit_transform(fake_followers_text)
print("Top words in fake_followers:", cv.get_feature_names_out())

Top words in TRAINED bot types: ['amp' 'blog' 'che' 'di' 'il' 'just' 'la' 'le' 'ma' 'new' 'non' 'post'
 'rt' 'se' 'si' 'u0e07' 'u0e23' 'u0e25' 'u0e2d' 'una']
Top words in fake_followers: ['el' 'en' 'la' 'los' 'para' 'que' 'rt' 'ð²ð' 'ðµ' 'ðµð' 'ðµð²ð¾ð' 'ðµñ'
 'ð¹ñ' 'ðºð' 'ð½ð' 'ð½ñ' 'ð¾' 'ð¾ð' 'ð¾ð½' 'ð¾ñ']


In [46]:
# Just look at columns for one of the folders — genuine_accounts is a good sample
sample_users = pd.read_csv(base_path + r'\genuine_accounts.csv\genuine_accounts.csv\users.csv', nrows=5)
print(sample_users.columns.tolist())
print(sample_users[['id']].head())  # adjust once we know the date column name

['id', 'name', 'screen_name', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'url', 'lang', 'time_zone', 'location', 'default_profile', 'default_profile_image', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'is_translator', 'follow_request_sent', 'protected', 'verified', 'notifications', 'description', 'contributors_enabled', 'following', 'created_at', 'timestamp', 'crawled_at', 'updated', 'test_set_1', 'test_set_2']
           id
0  1502026416
1  2492782375
2   293212315
3   191839658
4  3020965143


In [47]:
# Load created_at from each users.csv and merge into your text data
user_dates = []

for folder in folders_nrows.keys():
    path = f'{folder}/{folder}/users.csv'
    df = pd.read_csv(path, encoding='latin1')
    user_dates.append(df[['id', 'created_at']])

all_user_dates = pd.concat(user_dates, ignore_index=True)
all_user_dates.columns = ['user_id', 'created_at']

# Merge into your account-level text data
account_text = account_text.merge(all_user_dates, on='user_id', how='left')

print(account_text[['user_id', 'created_at']].head())
print(account_text['created_at'].isna().sum(), "accounts missing creation date")

   user_id                      created_at
0   678033  Mon Jan 22 01:57:38 +0000 2007
1   722623  Mon Jan 29 00:54:34 +0000 2007
2   755116  Tue Feb 06 05:25:49 +0000 2007
3   755746  Wed Feb 07 05:21:24 +0000 2007
4   785080  Wed Feb 21 01:08:16 +0000 2007
0 accounts missing creation date


In [2]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


In [4]:
import pandas as pd

folders_nrows = {
    'genuine_accounts.csv': (0, 600000),
    'traditional_spambots_1.csv': (1, 50000),
    'social_spambots_1.csv': (1, 50000),
    'social_spambots_2.csv': (1, 50000),
    'social_spambots_3.csv': (1, 50000),
    'fake_followers.csv': (1, 50000),
}

all_tweets = []
for folder, (label, n) in folders_nrows.items():
    path = f'{folder}/{folder}/tweets.csv'
    df = pd.read_csv(path, nrows=n, encoding='latin1')
    df['label'] = label
    df['account_type'] = folder
    all_tweets.append(df)

combined_tweets = pd.concat(all_tweets, ignore_index=True)
combined_tweets['text'] = combined_tweets['text'].fillna('')
print('Combined shape:', combined_tweets.shape)

C:\Users\almas\AppData\Local\Temp\ipykernel_18428\1538913888.py:15: DtypeWarning: Columns (0: place) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, nrows=n, encoding='latin1')


Combined shape: (850000, 27)


In [5]:
account_text = combined_tweets.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
account_text.columns = ['user_id', 'all_text']

labels = combined_tweets.groupby('user_id')['label'].first().reset_index()
account_text = account_text.merge(labels, on='user_id')

print(account_text.shape)

(703, 3)


In [6]:
account_text.to_csv('account_text_for_bilstm.csv', index=False)
print("Saved!")

Saved!


In [10]:
import pandas as pd

folders_nrows = {
    'genuine_accounts.csv': (0, 600000),
    'traditional_spambots_1.csv': (1, 50000),
    'social_spambots_1.csv': (1, 50000),
    'social_spambots_2.csv': (1, 50000),
    'social_spambots_3.csv': (1, 50000),
    'fake_followers.csv': (1, 50000),
}

all_tweets = []
for folder, (label, n) in folders_nrows.items():
    path = f'{folder}/{folder}/tweets.csv'
    df = pd.read_csv(path, nrows=n, encoding='latin1')
    df['label'] = label
    df['account_type'] = folder
    all_tweets.append(df)

combined_tweets = pd.concat(all_tweets, ignore_index=True)
combined_tweets['text'] = combined_tweets['text'].fillna('')
print('Combined shape:', combined_tweets.shape)

C:\Users\almas\AppData\Local\Temp\ipykernel_18428\1538913888.py:15: DtypeWarning: Columns (0: place) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, nrows=n, encoding='latin1')


Combined shape: (850000, 27)


In [14]:
account_text = combined_tweets.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
account_text.columns = ['user_id', 'all_text']

labels = combined_tweets.groupby('user_id')['label'].first().reset_index()
account_text = account_text.merge(labels, on='user_id')

print(account_text.shape)


(703, 3)


In [15]:
for folder in folders_nrows.keys():
    path = f'{folder}/{folder}/users.csv'
    df = pd.read_csv(path, encoding='latin1')
    print(folder, '→', df['created_at'].iloc[0])

genuine_accounts.csv → Tue Jun 11 11:20:35 +0000 2013
traditional_spambots_1.csv → 1183552203000L
social_spambots_1.csv → Tue Mar 17 08:51:12 +0000 2009
social_spambots_2.csv → Tue Mar 04 18:11:08 +0000 2014
social_spambots_3.csv → Sun Sep 14 11:20:09 +0000 2008
fake_followers.csv → Wed Oct 07 03:19:21 +0000 2009


In [16]:
user_dates = []

for folder in folders_nrows.keys():
    path = f'{folder}/{folder}/users.csv'
    df = pd.read_csv(path, encoding='latin1')
    
    if folder == 'traditional_spambots_1.csv':
        # This file uses epoch milliseconds with a trailing 'L' - clean and convert
        cleaned = df['created_at'].astype(str).str.replace('L', '', regex=False)
        df['created_at_clean'] = pd.to_datetime(cleaned.astype('int64'), unit='ms')
    else:
        # Normal text date format
        df['created_at_clean'] = pd.to_datetime(df['created_at'])
    
    user_dates.append(df[['id', 'created_at_clean']])

all_user_dates = pd.concat(user_dates, ignore_index=True)
all_user_dates.columns = ['user_id', 'created_at']

account_text = account_text.drop(columns=['created_at'], errors='ignore')  # remove old broken column if it exists
account_text = account_text.merge(all_user_dates, on='user_id', how='left')

print(account_text[['user_id', 'created_at']].head())
print(account_text['created_at'].isna().sum(), "missing dates")

C:\Users\almas\AppData\Local\Temp\ipykernel_18428\3071077857.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'])
C:\Users\almas\AppData\Local\Temp\ipykernel_18428\3071077857.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'])
C:\Users\almas\AppData\Local\Temp\ipykernel_18428\3071077857.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'])
C:\Users\almas\AppData\Local\Temp\ipykernel_18428\3071077857.py:

   user_id                 created_at
0   678033  2007-01-22 01:57:38+00:00
1   722623  2007-01-29 00:54:34+00:00
2   755116  2007-02-06 05:25:49+00:00
3   755746  2007-02-07 05:21:24+00:00
4   785080  2007-02-21 01:08:16+00:00
0 missing dates


In [18]:
user_dates = []

for folder in folders_nrows.keys():
    path = f'{folder}/{folder}/users.csv'
    df = pd.read_csv(path, encoding='latin1')
    
    if folder == 'traditional_spambots_1.csv':
        cleaned = df['created_at'].astype(str).str.replace('L', '', regex=False)
        df['created_at_clean'] = pd.to_datetime(cleaned.astype('int64'), unit='ms', utc=True)
    else:
        df['created_at_clean'] = pd.to_datetime(df['created_at'], utc=True)
    
    user_dates.append(df[['id', 'created_at_clean']])

all_user_dates = pd.concat(user_dates, ignore_index=True)
all_user_dates.columns = ['user_id', 'created_at']

account_text = account_text.drop(columns=['created_at'], errors='ignore')
account_text = account_text.merge(all_user_dates, on='user_id', how='left')

print(account_text['created_at'].min())
print(account_text['created_at'].max())

C:\Users\almas\AppData\Local\Temp\ipykernel_18428\2020562479.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'], utc=True)
C:\Users\almas\AppData\Local\Temp\ipykernel_18428\2020562479.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'], utc=True)
C:\Users\almas\AppData\Local\Temp\ipykernel_18428\2020562479.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at_clean'] = pd.to_datetime(df['created_at'], utc=True)
C:\Users\almas\AppData\Local\Temp\

2007-01-22 01:57:38+00:00
2013-12-09 02:05:32+00:00


In [19]:
print(account_text['created_at'].dt.year.value_counts().sort_index())

created_at
2007     40
2008     44
2009    283
2010    182
2011     16
2012     52
2013     86
Name: count, dtype: int64


In [20]:
cutoff = '2012-01-01'

train_data = account_text[account_text['created_at'] < cutoff]
test_data = account_text[account_text['created_at'] >= cutoff]

print("Train (older accounts):", train_data.shape)
print("Test (newer accounts):", test_data.shape)
print()
print("Train label balance:")
print(train_data['label'].value_counts())
print()
print("Test label balance:")
print(test_data['label'].value_counts())

Train (older accounts): (565, 4)
Test (newer accounts): (138, 4)

Train label balance:
label
1    449
0    116
Name: count, dtype: int64

Test label balance:
label
0    97
1    41
Name: count, dtype: int64


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Fit TF-IDF on TRAIN only (avoid data leakage)
vectorizer = TfidfVectorizer(max_features=20000, stop_words='english')
X_train = vectorizer.fit_transform(train_data['all_text'])
X_test = vectorizer.transform(test_data['all_text'])

y_train = train_data['label']
y_test = test_data['label']

model = SVC(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("Temporal Test Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Temporal Test Accuracy: 0.7246376811594203
              precision    recall  f1-score   support

           0       0.75      0.92      0.82        97
           1       0.58      0.27      0.37        41

    accuracy                           0.72       138
   macro avg       0.66      0.59      0.60       138
weighted avg       0.70      0.72      0.69       138



In [22]:
import json

temporal_results = {
    "test_type": "Temporal Split (train on accounts created 2007-2011, test on 2012-2013)",
    "train_size": int(train_data.shape[0]),
    "test_size": int(test_data.shape[0]),
    "train_label_balance": train_data['label'].value_counts().to_dict(),
    "test_label_balance": test_data['label'].value_counts().to_dict(),
    "accuracy": accuracy_score(y_test, predictions),
    "classification_report": classification_report(y_test, predictions, output_dict=True)
}

with open('text_temporal_results.json', 'w') as f:
    json.dump(temporal_results, f, indent=2)

print("Saved temporal test results!")
print(json.dumps(temporal_results, indent=2))

Saved temporal test results!
{
  "test_type": "Temporal Split (train on accounts created 2007-2011, test on 2012-2013)",
  "train_size": 565,
  "test_size": 138,
  "train_label_balance": {
    "1": 449,
    "0": 116
  },
  "test_label_balance": {
    "0": 97,
    "1": 41
  },
  "accuracy": 0.7246376811594203,
  "classification_report": {
    "0": {
      "precision": 0.7478991596638656,
      "recall": 0.9175257731958762,
      "f1-score": 0.8240740740740741,
      "support": 97.0
    },
    "1": {
      "precision": 0.5789473684210527,
      "recall": 0.2682926829268293,
      "f1-score": 0.36666666666666664,
      "support": 41.0
    },
    "accuracy": 0.7246376811594203,
    "macro avg": {
      "precision": 0.6634232640424591,
      "recall": 0.5929092280613528,
      "f1-score": 0.5953703703703703,
      "support": 138.0
    },
    "weighted avg": {
      "precision": 0.6977033376279572,
      "recall": 0.7246376811594203,
      "f1-score": 0.6881776704240472,
      "support": 138